# Neural Models (Learning, Depth, Activations, Output Layers)

Everything for this lab in one notebook: the write up, the code, the runs and the tests. Run all cells from top to bottom.

Needs `numpy` and `torch` (CPU build is fine): `pip install numpy torch`

## Task 1: Problem Specification

* Input space X = {0,1}², output space Y = {0,1}.
* Examples: (0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0.
* Sketch: the class 1 points (0,1) and (1,0) sit on one diagonal of the unit square, the class 0 points (0,0) and (1,1) on the other.
* **Why no straight line works:** a line w·x + b = 0 that puts (0,1) and (1,0) on the positive side needs w1 + b > 0 and w2 + b > 0, while (0,0) negative needs b < 0. Adding the first two gives w1 + w2 + 2b > 0, so w1 + w2 + b > −b > 0, which puts (1,1) on the positive side too. Contradiction.
* **Prediction for a single affine layer plus sigmoid:** it cannot classify all four. With binary cross entropy the best it can do is output 0.5 everywhere, loss ln 2 ≈ 0.693.

**Measured:** final loss 0.6931, probabilities [0.5, 0.5, 0.5, 0.5], 2/4 correct (the threshold 0.5 ≥ 0.5 labels everything 1). Prediction confirmed.

**Think about it:** XOR tests the claim that representation, not parameter count, decides what a model can express. A linear model could have a million inputs and still never compute XOR of two of them. What is needed is an intermediate representation in which the classes become linearly separable.

## Task 2: Model Design and Validation Criteria

* Architecture 2 → 2 → 1, hidden activation tanh (sigmoid and ReLU tried in Task 4D), output one logit, `BCEWithLogitsLoss` (sigmoid and binary cross entropy fused for numerical stability), Adam with learning rate 0.1, full batch, 3000 steps, `torch.manual_seed`.

1. **Why the hidden nonlinearity is necessary:** without it, W2(W1 x + b1) + b2 is a single affine map, so the network is no more powerful than the linear model above. The nonlinearity lets the hidden layer bend the input space so that XOR becomes linearly separable for the output unit.
2. **Why sigmoid plus BCE:** sigmoid maps a logit to a probability for a single yes/no output, and BCE is the negative log likelihood of a Bernoulli variable. Together the gradient with respect to the logit is simply p − y, which does not vanish when the sigmoid saturates on a wrong answer, unlike squared error.
3. **Success criteria:** (i) final loss well below ln 2, ideally under 0.01; (ii) all four thresholded predictions correct; (iii) first layer gradient nonzero at the start of training and the loss actually falls; (iv) behaviour checked across several seeds, not just one lucky run.

**Think about it:** the hidden units have no targets. What each one computes is determined entirely by backpropagation: the output error is passed back through W2 and the activation derivative, so each hidden unit is pushed towards whatever feature most reduces the final loss. The hidden features are a by product of minimising the output loss.

### Shared data and models

In [1]:
"""Shared pieces for Lab 5: XOR data and the 2-2-1 network."""
import torch
import torch.nn as nn

# The four labelled examples of the sensor-disagreement (XOR) problem
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
Y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

ACTIVATIONS = {"sigmoid": nn.Sigmoid, "tanh": nn.Tanh, "relu": nn.ReLU}


class XORNet(nn.Module):
    """2 inputs -> 2 hidden units (nonlinear) -> 1 output logit."""

    def __init__(self, activation="tanh", hidden=2, outputs=1):
        super().__init__()
        self.hidden = nn.Linear(2, hidden)   # W(1), b(1)
        self.act = ACTIVATIONS[activation]()
        self.out = nn.Linear(hidden, outputs)  # W(2), b(2)

    def forward(self, x):
        a1 = self.hidden(x)      # pre-activation  a(1) = W(1) x + b(1)
        h1 = self.act(a1)        # hidden output   h(1) = f(a(1))
        return self.out(h1)      # logits          a(2) = W(2) h(1) + b(2)


class LinearOnly(nn.Module):
    """Single affine map followed by sigmoid (inside the loss). Cannot solve XOR."""

    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(2, 1)

    def forward(self, x):
        return self.lin(x)

## Task 3: LLM Implementation

Prompt used:

> Generate minimal PyTorch code for the following model and dataset. Dataset: X = [[0,0],[0,1],[1,0],[1,1]], y = [0,1,1,0]. Model: 2 inputs, 2 hidden units with tanh activation, 1 output logit. Use BCEWithLogitsLoss, PyTorch default random initialisation, full batch training for 3000 steps with Adam (lr 0.1) on CPU. Do not change the architecture or task. After training, report the final loss, all four probabilities, thresholded labels, and one parameter gradient tensor (the first layer weight gradient). Set a random seed for reproducibility and explain each test in one sentence. Make the hidden activation a parameter so I can switch to sigmoid or ReLU.

Before running I located: forward pass `logits = model(X)`; scalar loss `loss = loss_fn(logits, Y)`; reverse mode AD `loss.backward()`; parameter update `opt.step()`.

Changes before execution: (1) moved the model into one shared cell so all experiments share exactly the same architecture; (2) added the per example gradient average check for Task 4B.

**Think about it:** from the code alone I can verify the architecture, loss choice, data, and that `zero_grad`, `backward`, `step` happen in the right order. Whether it actually learns, how sensitive it is to the seed, and the gradient magnitudes can only be found by running it.

### Training and evaluation

In [2]:
"""
Lab 5 Tasks 3 and 4A/4B: train the 2-2-1 XOR network with BCEWithLogitsLoss.

Prompt used (Task 3) is recorded in REPORT.md.
"""
import torch
import torch.nn as nn


def train(model, steps=3000, lr=0.1, verbose=True):
    loss_fn = nn.BCEWithLogitsLoss()   # sigmoid + binary cross-entropy, numerically stable
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for step in range(steps):
        opt.zero_grad()
        logits = model(X)              # forward pass
        loss = loss_fn(logits, Y)      # scalar loss (mean over the 4 examples)
        loss.backward()                # reverse-mode autodiff: fills p.grad = dL/dp
        opt.step()                     # optimiser updates the parameters
        history.append(loss.item())
        if verbose and step in (0, 1, 10, 100, 500, 1000, 2000, steps - 1):
            print(f"  step {step:5d}  loss {loss.item():.6f}")
    return history


def evaluate(model):
    with torch.no_grad():
        probs = torch.sigmoid(model(X))
    labels = (probs >= 0.5).float()
    return probs, labels, int((labels == Y).sum())


def run_xor(activation="tanh", seed=0):

    print("=== Task 1 check: single affine layer + sigmoid ===")
    torch.manual_seed(seed)
    lin = LinearOnly()
    h = train(lin, verbose=False)
    probs, labels, correct = evaluate(lin)
    print(f"  final loss {h[-1]:.4f} (ln 2 = 0.6931), correct {correct}/4")
    print("  probabilities:", [round(v, 3) for v in probs.squeeze().tolist()])

    print(f"\n=== 2-2-1 network, hidden activation = {activation}, seed = {seed} ===")
    torch.manual_seed(seed)
    net = XORNet(activation)

    # Task 4B: gradient on the very first backward pass, before any update
    loss = nn.BCEWithLogitsLoss()(net(X), Y)
    loss.backward()
    print("Initial dL/dW(1) (hidden.weight.grad):\n", net.hidden.weight.grad)
    # mean loss -> gradient is the average of the per-example gradients
    per_example = []
    for i in range(4):
        net.zero_grad()
        nn.BCEWithLogitsLoss()(net(X[i:i + 1]), Y[i:i + 1]).backward()
        per_example.append(net.hidden.weight.grad.clone())
    avg = torch.stack(per_example).mean(0)
    print("Average of the four per-example gradients:\n", avg)
    net.zero_grad()

    hist = train(net)
    probs, labels, correct = evaluate(net)
    print(f"\nInitial loss {hist[0]:.6f}   final loss {hist[-1]:.6f}")
    for x, pr, lb, y in zip(X.tolist(), probs.squeeze().tolist(), labels.squeeze().tolist(), Y.squeeze().tolist()):
        print(f"  x={x}  P(y=1)={pr:.4f}  predicted={int(lb)}  target={int(y)}")
    print(f"Correct: {correct}/4")

    net.zero_grad()
    nn.BCEWithLogitsLoss()(net(X), Y).backward()
    print("dL/dW(1) after training (should be near zero at a minimum):\n", net.hidden.weight.grad)

    print("\n=== Repeated runs (seeds 0..9) ===")
    ok = 0
    for s in range(10):
        torch.manual_seed(s)
        m = XORNet(activation)
        hh = train(m, verbose=False)
        c = evaluate(m)[2]
        ok += c == 4
        print(f"  seed {s}: final loss {hh[-1]:.5f}  correct {c}/4")
    print(f"Solved XOR in {ok}/10 seeds")

## Task 4: Results

### Part A: Basic learning (tanh, seed 0)

| | Value |
|---|---|
| Initial loss | 0.715159 |
| Final loss | 0.000038 |

| Input | P(y=1) | Predicted | Target |
|---|---|---|---|
| (0,0) | 0.0000 | 0 | 0 |
| (0,1) | 1.0000 | 1 | 1 |
| (1,0) | 0.9999 | 1 | 1 |
| (1,1) | 0.0000 | 0 | 0 |

4/4 correct. **Repeated runs:** over seeds 0 to 9, tanh solved XOR in 4/10. The failed runs mostly stop at loss 0.34658 = (ln 2)/2, which is a known local minimum of the 2 hidden unit XOR network: the model gets two examples perfectly and is stuck at 0.5 on the other two. This is a real optimisation property of such a tiny network, not a code bug. Seed 0 was kept as the reported run; I did not change the task to make other seeds pass.

### Part B: Backpropagation check

`net.hidden.weight.grad` after the first `backward()` is ∂L/∂W(1), the derivative of the mean loss with respect to each first layer weight:

```
tensor([[ 0.0005,  0.0006],
        [-0.0426, -0.0448]])
```

The average of the four separately computed per example gradients is identical. This is expected: the loss is the **mean** of the four example losses, and differentiation is linear, so ∂/∂W (1/4 Σ L_i) = 1/4 Σ ∂L_i/∂W. After training the gradient is about 10⁻⁶, consistent with sitting at a minimum.

### Run: tanh, seed 0 (Parts A and B)

In [3]:
run_xor(activation="tanh", seed=0)

=== Task 1 check: single affine layer + sigmoid ===


  final loss 0.6931 (ln 2 = 0.6931), correct 2/4
  probabilities: [0.5, 0.5, 0.5, 0.5]

=== 2-2-1 network, hidden activation = tanh, seed = 0 ===
Initial dL/dW(1) (hidden.weight.grad):
 tensor([[ 0.0005,  0.0006],
        [-0.0426, -0.0448]])
Average of the four per-example gradients:
 tensor([[ 0.0005,  0.0006],
        [-0.0426, -0.0448]])
  step     0  loss 0.715159
  step     1  loss 0.698722
  step    10  loss 0.665003
  step   100  loss 0.008251


  step   500  loss 0.000851


  step  1000  loss 0.000285


  step  2000  loss 0.000085


...

  seed 5: final loss 0.34658  correct 2/4


  seed 6: final loss 0.34658  correct 2/4


  seed 7: final loss 0.47739  correct 3/4


  seed 8: final loss 0.34658  correct 3/4


  seed 9: final loss 0.34658  correct 2/4
Solved XOR in 4/10 seeds


### Run: ReLU, seed 3

In [4]:
run_xor(activation="relu", seed=3)

=== Task 1 check: single affine layer + sigmoid ===


  final loss 0.6931 (ln 2 = 0.6931), correct 2/4
  probabilities: [0.5, 0.5, 0.5, 0.5]

=== 2-2-1 network, hidden activation = relu, seed = 3 ===
Initial dL/dW(1) (hidden.weight.grad):
 tensor([[0., 0.],
        [0., 0.]])
Average of the four per-example gradients:
 tensor([[0., 0.],
        [0., 0.]])
  step     0  loss 0.695374
  step     1  loss 0.693288
  step    10  loss 0.693581
  step   100  loss 0.693147


  step   500  loss 0.693147


  step  1000  loss 0.693147


  step  2000  loss 0.693147


...

  seed 5: final loss 0.47739  correct 3/4


  seed 6: final loss 0.69315  correct 2/4


  seed 7: final loss 0.47739  correct 3/4


  seed 8: final loss 0.00003  correct 4/4


  seed 9: final loss 0.69315  correct 2/4
Solved XOR in 2/10 seeds


### Part C: Symmetry experiment

**All parameters set to 0:** both rows of W(1) stayed at [0, 0] for all 2000 steps, loss stayed at 0.6931 and predictions were [1,1,1,1]. With zero weights, every hidden output is tanh(0) = 0, so the gradient on W(2) is zero; and since W(2) = 0, no error signal reaches W(1) either. The only nonzero candidate, the output bias, also gets zero gradient because the labels are balanced (mean of σ(0) − y is 0). Nothing ever moves.

**All parameters set to 0.5 (a nonzero but identical start):**

```
step    0  row0 [0.5, 0.5]        row1 [0.5, 0.5]        identical=True
step   50  row0 [0.6643, 0.6643]  row1 [0.6643, 0.6643]  identical=True
step 2000  row0 [3.9763, 3.9763]  row1 [3.9763, 3.9763]  identical=True
```

The weights change but the two rows remain exactly equal; final loss 0.4796, 3/4 correct. Two hidden units with the same incoming weights compute the same output, receive the same outgoing weight, and therefore get exactly the same gradient. They update identically forever, so the network effectively has one hidden unit, which is not enough for XOR. Random initialisation breaks this symmetry.

In [5]:
"""
Lab 5 Task 4C: symmetry experiment.  All weights and biases start at zero
(also shown: all equal to a nonzero constant). Hidden rows stay identical.
"""
import torch
import torch.nn as nn


def run_symmetry(init_value, steps=2000, activation="tanh"):
    torch.manual_seed(0)
    net = XORNet(activation)
    with torch.no_grad():
        for p in net.parameters():
            p.fill_(init_value)
    opt = torch.optim.SGD(net.parameters(), lr=0.5)
    loss_fn = nn.BCEWithLogitsLoss()
    print(f"\n--- all parameters initialised to {init_value}, activation {activation} ---")
    for step in range(steps + 1):
        opt.zero_grad()
        loss = loss_fn(net(X), Y)
        loss.backward()
        if step in (0, 1, 2, 5, 50, 500, steps):
            W = net.hidden.weight.detach()
            print(f"step {step:4d} loss {loss.item():.4f}  row0 {W[0].tolist()}  row1 {W[1].tolist()}"
                  f"  identical={torch.equal(W[0], W[1])}")
        opt.step()
    with torch.no_grad():
        preds = (torch.sigmoid(net(X)) >= 0.5).int().squeeze().tolist()
    print("final predictions:", preds, "targets:", Y.int().squeeze().tolist())


run_symmetry(0.0)
run_symmetry(0.5)


--- all parameters initialised to 0.0, activation tanh ---
step    0 loss 0.6931  row0 [0.0, 0.0]  row1 [0.0, 0.0]  identical=True
step    1 loss 0.6931  row0 [0.0, 0.0]  row1 [0.0, 0.0]  identical=True
step    2 loss 0.6931  row0 [0.0, 0.0]  row1 [0.0, 0.0]  identical=True
step    5 loss 0.6931  row0 [0.0, 0.0]  row1 [0.0, 0.0]  identical=True
step   50 loss 0.6931  row0 [0.0, 0.0]  row1 [0.0, 0.0]  identical=True


step  500 loss 0.6931  row0 [0.0, 0.0]  row1 [0.0, 0.0]  identical=True


step 2000 loss 0.6931  row0 [0.0, 0.0]  row1 [0.0, 0.0]  identical=True
final predictions: [1, 1, 1, 1] targets: [0, 1, 1, 0]

--- all parameters initialised to 0.5, activation tanh ---
step    0 loss 0.8522  row0 [0.5, 0.5]  row1 [0.5, 0.5]  identical=True
step    1 loss 0.7861  row0 [0.49672380089759827, 0.49672380089759827]  row1 [0.49672380089759827, 0.49672380089759827]  identical=True
step    2 loss 0.7457  row0 [0.4954908788204193, 0.4954908788204193]  row1 [0.4954908788204193, 0.4954908788204193]  identical=True
step    5 loss 0.6996  row0 [0.4983152747154236, 0.4983152747154236]  row1 [0.4983152747154236, 0.4983152747154236]  identical=True
step   50 loss 0.6722  row0 [0.6643314957618713, 0.6643314957618713]  row1 [0.6643314957618713, 0.6643314957618713]  identical=True


step  500 loss 0.4897  row0 [2.906039237976074, 2.906039237976074]  row1 [2.906039237976074, 2.906039237976074]  identical=True


step 2000 loss 0.4796  row0 [3.9763455390930176, 3.9763455390930176]  row1 [3.9763455390930176, 3.9763455390930176]  identical=True
final predictions: [0, 1, 1, 1] targets: [0, 1, 1, 0]


### Part D: Activation experiment

Same seed (0), same initial weights, only the hidden activation changes:

| Hidden activation | Final loss | 4/4 correct? | Early ‖∇W(1) L‖₂ (step 10) |
|---|---|---|---|
| Sigmoid | 0.47740 | no (3/4) | 0.00090 |
| Tanh | 0.00004 | yes | 0.03976 |
| ReLU | 0.69315 | no (2/4) | 0.00000 |

Across seeds 0 to 9: sigmoid 4/10 solved (mean early gradient norm 0.0042), tanh 4/10 (0.0356), ReLU 2/10 (0.0203); in 7 of the 10 ReLU runs at least one hidden unit was dead (negative pre activation on all four inputs).

Interpretation for **this** experiment (four points do not justify a general ranking):
* **Sigmoid** produced a gradient about 40 times smaller than tanh at the same step. The sigmoid derivative is at most 0.25 (tanh peaks at 1) and its outputs are not zero centred, so the signal reaching W(1) is weaker and learning is slower. With this seed it ended in the 3/4 local minimum.
* **Tanh** had the strongest early gradient and solved the task.
* **ReLU** had a gradient of exactly 0 for seed 0. At initialisation both units were active on some inputs (gradient norm 0.0017 at step 0), but within the first 3 Adam steps the large learning rate pushed both pre activations negative on every input. From then on their derivative is 0 and no learning signal ever reaches W(1) again. With only 2 hidden units, losing even one makes XOR unsolvable. A smaller learning rate or more hidden units would make this less likely.

**Think about it (two kinds of small gradient):** to tell saturation from dead ReLU, look at the pre activations a(1). A saturated sigmoid unit has large |a| (say above 5) and an output near 0 or 1, and its derivative is small but not zero. A dead ReLU unit has a ≤ 0 for every input, output exactly 0, and derivative exactly 0. The activation experiment counts dead units by checking `(a1 <= 0).all(dim=0)`.

In [6]:
"""
Lab 5 Task 4D: same random init (same seed), only the hidden activation changes.
Records final loss, 4/4 correct, and ||dL/dW(1)||_2 at an early step.
"""
import torch
import torch.nn as nn


EARLY_STEP = 10
STEPS = 3000
SEEDS = range(10)


def run_activation(activation, seed, lr=0.1):
    torch.manual_seed(seed)
    net = XORNet(activation)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    early_norm = None
    for step in range(STEPS):
        opt.zero_grad()
        loss = loss_fn(net(X), Y)
        loss.backward()
        if step == EARLY_STEP:
            early_norm = net.hidden.weight.grad.norm().item()
        opt.step()
    with torch.no_grad():
        a1 = net.hidden(X)
        probs = torch.sigmoid(net(X))
    correct = int(((probs >= 0.5).float() == Y).sum())
    dead = int((a1 <= 0).all(dim=0).sum()) if activation == "relu" else 0
    return loss.item(), correct, early_norm, dead


def activation_tables():
    print(f"Seed 0 (the run used in the report table), Adam lr=0.1, {STEPS} steps\n")
    print("| Hidden activation | Final loss | 4/4 correct? | ||grad W(1)|| at step 10 |")
    print("|---|---|---|---|")
    for act in ["sigmoid", "tanh", "relu"]:
        loss, correct, norm, _ = run_activation(act, 0)
        print(f"| {act} | {loss:.5f} | {'yes' if correct == 4 else f'no ({correct}/4)'} | {norm:.5f} |")

    print(f"\nRobustness over seeds {list(SEEDS)}\n")
    print("| Hidden activation | Seeds solved | Mean early grad norm | Runs with a dead ReLU unit |")
    print("|---|---|---|---|")
    for act in ["sigmoid", "tanh", "relu"]:
        rs = [run_activation(act, s) for s in SEEDS]
        solved = sum(r[1] == 4 for r in rs)
        mean_norm = sum(r[2] for r in rs) / len(rs)
        dead = sum(r[3] > 0 for r in rs)
        print(f"| {act} | {solved}/{len(rs)} | {mean_norm:.5f} | {dead if act == 'relu' else 'n/a'} |")


activation_tables()

Seed 0 (the run used in the report table), Adam lr=0.1, 3000 steps

| Hidden activation | Final loss | 4/4 correct? | ||grad W(1)|| at step 10 |
|---|---|---|---|


| sigmoid | 0.47740 | no (3/4) | 0.00090 |


| tanh | 0.00004 | yes | 0.03976 |


| relu | 0.69315 | no (2/4) | 0.00000 |

Robustness over seeds [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

| Hidden activation | Seeds solved | Mean early grad norm | Runs with a dead ReLU unit |
|---|---|---|---|


| sigmoid | 4/10 | 0.00417 | n/a |


| tanh | 4/10 | 0.03558 | n/a |


| relu | 2/10 | 0.02026 | 7 |


## Task 5: Three Class Extension

Predictions before running: (1) final weight matrix W(2) has shape 3 × 2 (3 classes, 2 hidden units); (2) 3 logits per example, so the logits tensor is 4 × 3; (3) softmax divides each exp(z_k) by their sum, so the outputs are positive and sum to 1 by construction; (4) with L = −log p_y and p = softmax(z), ∂L/∂z_k = p_k − 1[k = y], i.e. p − y with y one hot.

Change requested from the LLM: replace the output layer with `nn.Linear(2, 3)` and the loss with `nn.CrossEntropyLoss()`, nothing else. Measured:

| Input | P(class 0, 1, 2) | Argmax | Target |
|---|---|---|---|
| (0,0) | [1.0, 0.0, 0.0] | 0 | 0 |
| (0,1) | [0.0, 1.0, 0.0] | 1 | 1 |
| (1,0) | [0.0, 1.0, 0.0] | 1 | 1 |
| (1,1) | [0.0, 0.0, 1.0] | 2 | 2 |

Final loss 0.000014, shapes confirmed: W(2) is (3, 2), logits (4, 3). Softmax for x = (0,1): [8.33e−06, 0.999989, 2.63e−06], sum = 1.00000000.

**p − y check** (on the untrained network so the numbers are not tiny):

```
autograd dL/dz : [0.239472, -0.669796, 0.430324]
p - y          : [0.239472, -0.669796, 0.430324]
```

**Shift invariance:** adding 100 to all logits changed the probabilities by at most 3.3e−11 (floating point roundoff), because exp(z + c) / Σ exp(z + c) = exp(z) / Σ exp(z). Adding 1000 and using the naive formula gives `[nan, nan, nan]` (exp overflows to infinity, and inf / inf is nan), while subtracting the max logit first gives the correct vector. That is why stable implementations compute exp(z − max z): it is mathematically the same function but the largest exponent becomes exp(0) = 1.

**Think about it:** for next token prediction, the softmax, the cross entropy loss, the p − y gradient and the max subtraction trick are exactly the same with 50,000 classes as with 3. What changes is everything around it: the inputs become token sequences, the hidden layers become deep transformer blocks with attention over long contexts, the output matrix becomes huge (hidden size × vocabulary), and training needs massive data, mini batches and GPUs.

In [7]:
"""
Lab 5 Task 5: three-class sensor decision.
class 0 = both off (0,0), class 1 = disagree (0,1)/(1,0), class 2 = both on (1,1)
Only the output layer and loss change: 3 logits + CrossEntropyLoss.
"""
import torch
import torch.nn as nn


Y3 = torch.tensor([0, 1, 1, 2])


def check_p_minus_y(net, label):
    """Compare autograd's dL/dlogits with p - y for the example x=(0,1)."""
    with torch.no_grad():
        z0 = net(X[1:2])
    z = z0.clone().requires_grad_(True)
    nn.CrossEntropyLoss()(z, Y3[1:2]).backward()
    onehot = torch.nn.functional.one_hot(Y3[1:2], 3).float()
    print(f"[{label}] autograd dL/dz :", [round(v, 6) for v in z.grad[0].tolist()])
    print(f"[{label}] p - y          :", [round(v, 6) for v in (torch.softmax(z0, 1) - onehot)[0].tolist()])


def run_three_class():
    torch.manual_seed(0)
    net = XORNet("tanh", hidden=2, outputs=3)
    check_p_minus_y(net, "before training")
    print("Final weight matrix shape W(2):", tuple(net.out.weight.shape), "(3 classes x 2 hidden)")
    loss_fn = nn.CrossEntropyLoss()  # applies log-softmax internally
    opt = torch.optim.Adam(net.parameters(), lr=0.1)
    for step in range(3000):
        opt.zero_grad()
        loss = loss_fn(net(X), Y3)
        loss.backward()
        opt.step()
    print(f"Final loss {loss.item():.6f}")

    with torch.no_grad():
        logits = net(X)
        probs = torch.softmax(logits, dim=1)
    print("Logits per example:", tuple(logits.shape))
    for x, p, y in zip(X.tolist(), probs.tolist(), Y3.tolist()):
        print(f"  x={x}  P={[round(v, 4) for v in p]}  argmax={p.index(max(p))}  target={y}")

    p0 = probs[1]
    print(f"\nSoftmax vector for x=(0,1): {p0.tolist()}  sum = {p0.sum().item():.8f}")

    check_p_minus_y(net, "after training")

    # Shift invariance and the max-subtraction trick
    shifted = torch.softmax(logits + 100.0, dim=1)
    print("\nmax |softmax(z+100) - softmax(z)| =", (shifted - probs).abs().max().item())
    big = logits[1] + 1000.0
    naive = torch.exp(big) / torch.exp(big).sum()
    stable = torch.exp(big - big.max()) / torch.exp(big - big.max()).sum()
    print("naive softmax of z+1000 :", naive.tolist())
    print("stable softmax of z+1000:", stable.tolist())


run_three_class()

[before training] autograd dL/dz : [0.239472, -0.669796, 0.430324]
[before training] p - y          : [0.239472, -0.669796, 0.430324]
Final weight matrix shape W(2): (3, 2) (3 classes x 2 hidden)


Final loss 0.000014
Logits per example: (4, 3)
  x=[0.0, 0.0]  P=[1.0, 0.0, 0.0]  argmax=0  target=0
  x=[0.0, 1.0]  P=[0.0, 1.0, 0.0]  argmax=1  target=1
  x=[1.0, 0.0]  P=[0.0, 1.0, 0.0]  argmax=1  target=1
  x=[1.0, 1.0]  P=[0.0, 0.0, 1.0]  argmax=2  target=2

Softmax vector for x=(0,1): [8.326723218488041e-06, 0.9999890327453613, 2.6256300316163106e-06]  sum = 1.00000000
[after training] autograd dL/dz : [8e-06, -1.1e-05, 3e-06]
[after training] p - y          : [8e-06, -1.1e-05, 3e-06]

max |softmax(z+100) - softmax(z)| = 5.093170329928398e-11
naive softmax of z+1000 : [nan, nan, nan]
stable softmax of z+1000: [8.326595889229793e-06, 0.9999890327453613, 2.625682554935338e-06]


### Tests

In [8]:
"""Fast checks for Lab 5.  Run: python -m unittest -v"""
import unittest

import torch
import torch.nn as nn


class TestXOR(unittest.TestCase):
    def test_linear_model_cannot_solve_xor(self):
        torch.manual_seed(0)
        m = LinearOnly()
        train(m, steps=1000, verbose=False)
        self.assertLess(evaluate(m)[2], 4)

    def test_tanh_seed0_solves_xor(self):
        torch.manual_seed(0)
        m = XORNet("tanh")
        hist = train(m, steps=1000, verbose=False)
        self.assertEqual(evaluate(m)[2], 4)
        self.assertLess(hist[-1], hist[0])

    def test_gradient_is_nonzero_initially(self):
        torch.manual_seed(0)
        m = XORNet("tanh")
        nn.BCEWithLogitsLoss()(m(X), Y).backward()
        self.assertGreater(m.hidden.weight.grad.norm().item(), 0)

    def test_symmetric_init_keeps_rows_identical(self):
        m = XORNet("tanh")
        with torch.no_grad():
            for p in m.parameters():
                p.fill_(0.5)
        opt = torch.optim.SGD(m.parameters(), lr=0.5)
        for _ in range(200):
            opt.zero_grad()
            nn.BCEWithLogitsLoss()(m(X), Y).backward()
            opt.step()
        W = m.hidden.weight
        self.assertTrue(torch.equal(W[0], W[1]))

    def test_softmax_sums_to_one_and_is_shift_invariant(self):
        z = torch.randn(4, 3)
        p = torch.softmax(z, 1)
        self.assertTrue(torch.allclose(p.sum(1), torch.ones(4)))
        self.assertTrue(torch.allclose(torch.softmax(z + 100, 1), p, atol=1e-6))


unittest.main(argv=["notebook"], exit=False, verbosity=2)

test_gradient_is_nonzero_initially (__main__.TestXOR.test_gradient_is_nonzero_initially) ... 

ok


test_linear_model_cannot_solve_xor (__main__.TestXOR.test_linear_model_cannot_solve_xor) ... 

ok


test_softmax_sums_to_one_and_is_shift_invariant (__main__.TestXOR.test_softmax_sums_to_one_and_is_shift_invariant) ... 

...

test_symmetric_init_keeps_rows_identical (__main__.TestXOR.test_symmetric_init_keeps_rows_identical) ... 

ok


test_tanh_seed0_solves_xor (__main__.TestXOR.test_tanh_seed0_solves_xor) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 1.369s

OK


## Reflection Questions

1. **Depth vs nonlinearity:** stacking affine layers alone adds parameters but not expressive power; the linear model stayed at loss ln 2. Two layers with a nonlinear hidden activation solved XOR. Nonlinearity is what matters; depth only helps once there is a nonlinearity between layers.
2. **Useful learning signal, not just nonzero:** loss fell from 0.715 to 0.000038, all four predictions became correct, and the gradient shrank to about 10⁻⁶ at the end. A nonzero gradient that does not reduce the loss (as in the 0.5 identical start, stuck at 0.48) would not count.
3. **Identical initialisation:** identical units compute identical outputs and receive identical gradients, so they stay identical. The two hidden units collapse into one feature. With all zeros, no gradient reaches W(1) at all.
4. **Activation and gradient:** engineering observation: at step 10 the first layer gradient norm was 0.0009 (sigmoid), 0.040 (tanh), 0.000 (ReLU, seed 0). Scientific explanation: the gradient reaching W(1) is multiplied by f′(a); sigmoid′ ≤ 0.25 shrinks it, tanh′ ≤ 1 shrinks it less, and ReLU′ is exactly 0 for negative inputs, so a dead unit passes nothing back.
5. **Output layer and loss together:** the output activation defines what the network's numbers mean (one probability, or a distribution over K classes), and the loss must be the matching negative log likelihood. Matched pairs give the clean p − y gradient; mismatched pairs (e.g. softmax over a single output, or MSE on probabilities) give wrong semantics or vanishing gradients.
6. **LLM productivity vs human verification:** the LLM produced the training loop, the argument parsing and the softmax diagnostics in minutes. Human verification was essential in interpreting the repeated seed results: 6/10 seeds failing could look like a bug, but checking the loss value (exactly (ln 2)/2) showed it is a known local minimum. Also, my first p − y check was run on the trained network, where both sides were around 10⁻⁵ and the comparison proved little; I moved it to the untrained network.
7. **Tests at scale:** keep: loss decreasing, predictions on a held out set, gradient norms per layer, dead unit and saturation monitoring, symmetry breaking (by design), softmax sums to 1, shift invariance and p − y on a small batch. Too expensive: exhaustive finite difference gradient checks for every parameter (two forward passes per parameter), training across many seeds, and printing full gradient tensors. These would be replaced by spot checks on a few random parameters or a small model.